# Coordinates: from homology classes to maps

`tda.coords` turns persistent cohomology classes into coordinates on the
spaces they classify: H1 → circle (`circular`), several H1 classes → torus
(still `circular`), H1 mod 2 → real projective space (`projective`),
H2 → complex projective space (`complex_projective`). Every data point gets a
value; the class is chosen automatically (most persistent first).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tda

rng = np.random.default_rng(1)

## Circular coordinates: one H1 class → angles

The dominant class of a noisy circle should recover the angle, up to
rotation and direction.

In [ ]:
theta = rng.uniform(0, 2 * np.pi, 400)
points = np.stack([np.cos(theta), np.sin(theta)], axis=1)
points += rng.normal(scale=0.05, size=points.shape)

res = tda.coords.circular(points)
angles = res.coordinates[:, 0]
print('class interval:', res.intervals[0], '  computed at scale:', res.scales[0])

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 4))
ax0.scatter(*points.T, c=angles, cmap='hsv', s=8)
ax0.set_aspect('equal'); ax0.set_title('points colored by circular coordinate')
ax1.scatter(theta, angles, s=4)
ax1.set_xlabel('true angle'); ax1.set_ylabel('circular coordinate')
plt.tight_layout()

## Toroidal: several classes at once

Ask for more classes and each gets a column. Classes alive at a common scale
are decorrelated jointly (`res.groups` reports the blocks) — on two side-by-side
circles, each column tracks exactly one circle and stays flat on the other.

In [ ]:
shift = points + np.array([4.0, 0.0])
two = np.concatenate([points, shift])

res = tda.coords.circular(two, n_classes=2)
print('toroidal blocks:', res.groups)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for column, ax in enumerate(axes):
    ax.scatter(*two.T, c=res.coordinates[:, column], cmap='hsv', s=8)
    ax.set_aspect('equal'); ax.set_title(f'column {column}')
plt.tight_layout()

## Complex projective: an H2 class → the Riemann sphere

For a sphere sample, the dominant H2 class maps into CP¹, which *is* the
sphere — the composition recovers the sphere up to rotation.

In [ ]:
z = 1 - 2 * (np.arange(200) + 0.5) / 200
r = np.sqrt(1 - z * z)
golden = np.pi * (3 - np.sqrt(5))
sphere = np.stack([r * np.cos(golden * np.arange(200)),
                   r * np.sin(golden * np.arange(200)), z], axis=1)

res = tda.coords.complex_projective(sphere)
a, b = res.coordinates[:, 0], res.coordinates[:, 1]
image = np.stack([np.abs(a)**2 - np.abs(b)**2,
                  2 * (np.conj(a) * b).real,
                  2 * (np.conj(a) * b).imag], axis=1)

fig = plt.figure(figsize=(9, 4))
for i, (data, title) in enumerate([(sphere, 'input'), (image, 'image in CP¹ ≅ S²')]):
    ax = fig.add_subplot(1, 2, i + 1, projection='3d')
    ax.scatter(*data.T, c=sphere[:, 2], s=8)
    ax.set_title(title)
plt.tight_layout()

## Real projective coordinates, and going faster

`tda.coords.projective` maps an H1 mod-2 class into RP^d (unit vectors up to
sign) — the same call shape as everything above. And every coordinates
function accepts `quotient=True` to run on the quotient-cover approximation:
much faster, per-cluster values.

In [ ]:
rp = tda.coords.projective(points)
print('RP^2 embedding:', rp.coordinates.shape, ' class:', rp.interval)

fast = tda.coords.circular(points, quotient=True)
plt.figure(figsize=(4.5, 4))
plt.scatter(*points.T, c=fast.coordinates[:, 0], cmap='hsv', s=8)
plt.gca().set_aspect('equal')
plt.title('circular coordinate through the quotient')
plt.tight_layout()

## Fit once, transform new data

`circular_map` fits the same circular coordinates but keeps the model, so
new points can be mapped into the learned coordinate without recomputing
anything — sklearn-style. Points too far from the training data (outside
the cover) come back as NaN.

In [ ]:
model = tda.coords.circular_map(points)

fresh_theta = rng.uniform(0, 2 * np.pi, 250)
fresh = np.stack([np.cos(fresh_theta), np.sin(fresh_theta)], axis=1)
fresh += rng.normal(scale=0.05, size=fresh.shape)
new_angles = model.transform(fresh)[:, 0]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 4))
ax0.scatter(*points.T, c=np.asarray(model.coordinates)[:, 0], cmap='hsv', s=8)
ax0.set_aspect('equal'); ax0.set_title('training data (fit)')
ax1.scatter(*fresh.T, c=new_angles, cmap='hsv', s=8)
ax1.set_aspect('equal'); ax1.set_title('new data (transform)')
plt.tight_layout()